# 01 · Exploring light curves

What a Kepler/TESS light curve looks like, what cleaning and detrending do to it, and what the processed training set looks like per class.

Runs offline: if MAST is unreachable it falls back to a synthetic star.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from transit_hunter.config import load_data_config, setup_logging

setup_logging("WARNING")
cfg = load_data_config()
plt.rcParams.update({"figure.figsize": (12, 3.5), "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#199e70"

In [ ]:
TARGET = "Kepler-10"   # try: "Kepler-7", "TIC 261136679" (Pi Mensae), "KIC 11904151"
MISSION = None         # "Kepler", "TESS" or None (auto)
MAX_SEGMENTS = 4

## 1. Download (or load from cache)

In [ ]:
from transit_hunter.data.fetch import fetch_lightcurve
from transit_hunter.data.synthetic import synthetic_planet_lightcurve

try:
    lc = fetch_lightcurve(TARGET, mission=MISSION, max_segments=MAX_SEGMENTS)
    time, flux, flux_err = lc.time, lc.flux, lc.flux_err
    label = f"{lc.target_id} ({lc.mission}, {lc.author}, segments {lc.segments})"
except Exception as exc:  # offline, MAST down, ...
    print(f"Could not fetch {TARGET!r} ({type(exc).__name__}: {exc}).\nUsing a synthetic Kepler-10b-like light curve instead.")
    syn = synthetic_planet_lightcurve(period=0.8375, t0=0.3, depth=0.00025, duration=0.075,
                                      baseline_days=120, noise_ppm=80, variability_ppm=600, seed=1)
    time, flux, flux_err = syn.time, syn.flux, syn.flux_err
    label = "synthetic Kepler-10b-like star"
print(label, f"| {time.size:,} points over {time.max() - time.min():.1f} days")

In [ ]:
plt.plot(time, (flux - 1) * 1e6, ".", ms=1.5, color=BLUE)
plt.xlabel("Time (days, mission time system)"); plt.ylabel("Flux change (ppm)")
plt.title(f"Raw light curve: {label}"); plt.show()

## 2. Cleaning and detrending

Cleaning clips *upward* outliers only (transits are downward). The biweight trend should follow the slow wiggles and ignore the short dips. Savitzky–Golay is shown for comparison.

In [ ]:
from transit_hunter.processing.clean import clean_lightcurve, robust_std
from transit_hunter.processing.detrend import detrend

c = clean_lightcurve(time, flux, flux_err, cfg.cleaning)
print(f"removed {c.n_nan_removed} NaNs and {c.n_outliers_removed} upward outliers")
bw = detrend(c.time, c.flux, c.flux_err, cfg.detrend, method="biweight")
sg = detrend(c.time, c.flux, c.flux_err, cfg.detrend, method="savgol")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(c.time, (c.flux - 1) * 1e6, ".", ms=1.5, color="0.6", label="cleaned flux")
axes[0].plot(bw.time, (bw.trend - 1) * 1e6, color=ORANGE, lw=1.5, label="biweight trend")
axes[0].plot(sg.time, (sg.trend - 1) * 1e6, color=AQUA, lw=1, ls="--", label="Savitzky-Golay trend")
axes[0].set_ylabel("ppm"); axes[0].legend(loc="upper right")
axes[1].plot(bw.time, (bw.flux - 1) * 1e6, ".", ms=1.5, color=BLUE)
axes[1].set_ylabel("ppm"); axes[1].set_xlabel("Time (days)"); axes[1].set_title("Detrended (biweight)")
plt.tight_layout(); plt.show()

for name, d in (("raw", c), ("biweight", bw), ("savgol", sg)):
    print(f"{name:>9}: point-to-point scatter {robust_std(np.diff(d.flux)) / np.sqrt(2) * 1e6:7.1f} ppm, "
          f"overall scatter {robust_std(d.flux) * 1e6:7.1f} ppm")

## 3. The processed training set

Run `make data-process` (or `make data`) first. Each class gets its average global and local view: planets should show a clean U, EBs a deeper V plus a secondary at phase ±0.5, other false positives a weak, messy dip.

In [ ]:
from transit_hunter.config import CLASSES, CLASS_NAMES
from transit_hunter.data.dataset import load_dataset

try:
    ds = load_dataset()
except FileNotFoundError:
    ds = None
    print("No processed dataset yet; run `make data-process`.")

if ds is not None:
    print(ds.index.groupby(["split", "label"]).size().unstack(fill_value=0))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for cls, color in zip(CLASSES, (BLUE, ORANGE, AQUA)):
        m = (ds.index["label"] == cls).to_numpy()
        if m.any():
            axes[0].plot(np.median(ds.global_views[m], axis=0), color=color, label=f"{CLASS_NAMES[cls]} (n={m.sum()})")
            axes[1].plot(np.median(ds.local_views[m], axis=0), color=color)
    axes[0].set_title("Median global view (201 bins)"); axes[1].set_title("Median local view (61 bins)")
    axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
if ds is not None:
    feats = ["log_depth", "shape_ratio", "secondary_snr", "odd_even_sigma", "duration_ratio", "snr"]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, f in zip(axes.ravel(), feats):
        vals = ds.index[f].astype(float)
        lo, hi = np.nanpercentile(vals, [1, 99])
        for cls, color in zip(CLASSES, (BLUE, ORANGE, AQUA)):
            v = vals[ds.index["label"] == cls].dropna()
            ax.hist(v.clip(lo, hi), bins=30, range=(lo, hi), histtype="step", lw=1.5, color=color, label=cls, density=True)
        ax.set_title(f)
    axes[0, 0].legend(); plt.tight_layout(); plt.show()